In [ ]:
import pandas as pd
import numpy as np
import math
import re
import os
import matplotlib.pylab as plt
import seaborn as sns
import scipy.io as sio
from tqdm import tqdm

# import functions

In [ ]:
def deal_withMATstruct(df):

    #deal with Experimet column
    experiment = [int(x[0][0]) for x in np.array(df.Experiment)]

    #deal with Expweriment Name column
    exp_name = [str(x[0][0]) for x in np.array(df.ExperimentName)]

    #deal with Series number
    series_num = [int(x[0][0]) for x in np.array(df.Rec)]

    #deal with Rseries values
    try:
        rseries = [float(x[0][0]) for x in np.array(df.Rs)]
    except:
        rseries = [x[0][0] for x in np.array(df.Rs)]
    #dela with Cm
    try:
        cmvals = [float(x[0][0]) for x in np.array(df.Cm)]
    except:
        cmvals = [x[0][0] for x in np.array(df.Cm)]
    
    #deal with nSweeps
    nsweeps = [int(x[0][0]) for x in np.array(df.nSweeps)]

    #deal with Vhold
    try:
        vhold = [float(x[0][0]) for x in np.array(df.Vhold)]
    except:
        vhold = [x[0][0] for x in np.array(df.Vhold)]

    #uncompensated R-series
    try:
        rseries_uncomp = [float(x[0][0]) for x in np.array(df.Rs_uncomp)]
    except:
        rseries_uncomp = [x[0][0] for x in np.array(df.Rs_uncomp)]

    #Rs faction compensation.. (we dont really use this)
    try:
        rseries_fraction = [float(x[0][0]) for x in np.array(df.RsFractionComp)]
    except:
        rseries_fraction = [x[0][0] for x in np.array(df.RsFractionComp)]

    #series name
    series_name = [str(x[0]) for x in np.array(df.Stimulus)]

    #time unit
    tunit = [str(x[0][0][0]) for x in np.array(df.TimeUnit)]

    #rec units
    rec_units = [str(x[0][0][0]) for x in np.array(df.ChUnit)]

    #rec type
    rec_type = [str(x[0][0][0]) for x in np.array(df.ChName)]

    #rec mnode (basically rec type but verbose)
    rec_mode = [str(x[0][0][0]) for x in np.array(df.RecMode)]

    #temperature
    temper = [float(x[0][0]) for x in np.array(df.Temperature)]

    # external_sol (unused, all nans)
    ext_sol = [float(x[0][0]) for x in np.array(df.ExternalSolution)]

    #InternalSolution
    int_sol = [float(x[0][0]) for x in np.array(df.InternalSolution)]

    #Sampling Rate
    sr = [float(x[0][0]) for x in np.array(df.SR)]

    #TimeStam.. skipping since the data is gibberish
    #timestamp = []
    
    # RAW DATA
    raw_data = [x[0][0] for x in np.array(df.dataRaw)]

    # get pulse data
    pulse_data = [x[0][0][0] for x in np.array(df.stimWave)]

    ndf = pd.DataFrame()

    ndf['experiment'] = experiment
    ndf['exp_name'] = exp_name 
    ndf['series_num'] = series_num 
    ndf['rseries'] = rseries 
    ndf['Cmvals'] = cmvals 
    ndf['Nsweeps'] = nsweeps 
    ndf['Vhold'] = vhold
    ndf['Rseries_uncomp'] =  rseries_uncomp 
    ndf['Rseries_frac'] = rseries_fraction 
    ndf['series_name'] = series_name 
    ndf['time_unit'] = tunit
    ndf['rec_units'] = rec_units 
    ndf['rec_type'] = rec_type 
    ndf['rec_mode'] = rec_mode
    ndf['temperature'] = temper 
    ndf['external_sol'] = ext_sol 
    ndf['internal_sol'] = int_sol
    ndf['SR'] = sr 
    ndf['rawData'] = [x for x in raw_data]
    ndf['pulseData'] = [x for x in pulse_data]
    return ndf

In [ ]:
def get_stepInfo(self):
    asteps = []
    for i in range(len(self)):
        steps = []
        pulses = self.iloc[i].pulseData
        for j in range(pulses.shape[1]):
            steps.append(round(pulses[:,j][len(pulses[:,j])//2-100], 4))
        asteps.append(steps)

    self['current_Steps'] = [x for x in asteps]
    return self


def process_HEKAmat_object(mfile):
    global fpath
    fpath = mfile
    mfile = sio.loadmat(os.path.join(mfile))
    
    for i in range(1000):  #this loop is capped at 1000.. it may be better to use a while loop here..
        try:                         #essentially, to get the data from the structure, one needs to be able to index it
            pd.DataFrame(mfile['S'][i])  #however, there is no way top check the structure shape/length.. this loop queries this without crashing the code
        except:
            breaker = i
            break

    dfs = []
    for i in range(breaker):
        dfs.append(pd.DataFrame(mfile['S'][i]))

    df = pd.concat(dfs).reset_index(drop=True)

    tmp = deal_withMATstruct(df)
    tmp = get_stepInfo(tmp)
    tmp['fpath'] = fpath

    return tmp
    

In [ ]:
def substring_search(s):
    end = fpath.find('struct')

    i=end
    while i > 0:
        if fpath[i] == '\\':
            start = i
            break
        i-=1

    return fpath[start+1:end]


def separateNsave(adf, out_path = 'Z:\\'):


    for f in adf.fpath.unique():
        tmp = adf[adf.fpath==f].reset_index(drop=True)
        date = substring_search(f)
        tmp.to_pickle(os.path.join(out_path, date + '_heka_structure.pkl'))
    return


# sample usage

In [ ]:
input_path = r'J:\mpoa_patch_1-29-26'

mfiles = []
for f in os.listdir(input_path):
    if '.mat' in f:
        mfiles.append(f)

dfs = []
k=0
for idx, m in enumerate(tqdm(mfiles, desc = 'processing file '+str(k+1)+'/'+str(len(mfiles)))):
    mfile = os.path.join(input_path, m)
    df = process_HEKAmat_object(mfile)
    dfs.append(df)
    k+=1
    
adf = pd.concat(dfs).reset_index(drop=True)




In [ ]:
separateNsave(adf, out_path = r'J:\mpoa_patch_1-29-26')